## <span style="color:PURPLE">PACKAGES USED</span> ##

In [3]:
# ============================================================
# PACKAGES USED
# ============================================================

from pathlib import Path
import warnings

import base64
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

from IPython.display import display

import plotly.graph_objects as go

# <span style="color:PURPLE"> RELATIONSHIPS WHITIN FEATURE GROUPS BINARY_ENCODING </span>

## <span style="color:PURPLE"> BINARY ENCODING </span> ##

In [6]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "binary_encoding"
)

FEATURE_1 = (
    "SEND_GENDER_BE"
)

FEATURE_2 = (
    "TRANS_YEAR_BE"
)

VALID_FEATURE_1_VALUES = [
    "F",
    "M"
]

VALID_FEATURE_2_VALUES = [
    2019,
    2020
]

FEATURE_1_BINARY_MAPPING = {
    "F": 0,
    "M": 1
}

FEATURE_2_BINARY_MAPPING = {
    2019: 0,
    2020: 1
}

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_binary_encoding.html"
)


GROUPED_PROPORTIONS_PATH = (
    RESULTS_DIRECTORY
    / "binary_encoding_grouped_proportions.png"
)


# ============================================================
# 06. REMOVE OBSOLETE OUTPUT FILES
# ============================================================

OBSOLETE_OUTPUTS = [
    RESULTS_DIRECTORY
    / "binary_encoding_count_heatmap.png",

    RESULTS_DIRECTORY
    / "binary_encoding_percentage_heatmap.png"
]


for obsolete_output in OBSOLETE_OUTPUTS:

    if obsolete_output.exists():

        obsolete_output.unlink()


# ============================================================
# 07. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 08. LOAD ONLY THE FEATURES BEING ANALYZED
# ============================================================

dataset_features = pd.read_parquet(
    DATASET_PATH,
    columns=[
        FEATURE_1,
        FEATURE_2
    ]
)


# ============================================================
# 09. BASIC DATASET OVERVIEW
# ============================================================

total_observations = int(
    len(
        dataset_features
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


feature_1_missing = int(
    dataset_features[
        FEATURE_1
    ]
    .isna()
    .sum()
)


feature_2_missing = int(
    dataset_features[
        FEATURE_2
    ]
    .isna()
    .sum()
)


pair_missing = int(
    dataset_features[
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .isna()
    .any(
        axis=1
    )
    .sum()
)


pair_missing_percentage = (
    pair_missing
    / total_observations
    * 100
)


valid_pair = (
    dataset_features[
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .dropna()
    .copy()
)


valid_observations = int(
    len(
        valid_pair
    )
)


if valid_observations == 0:

    raise ValueError(
        "No complete feature pairs are available."
    )


# ============================================================
# 10. OBSERVED VALUES
# ============================================================

feature_1_unique_values = sorted(
    valid_pair[
        FEATURE_1
    ]
    .unique()
    .tolist()
)


feature_2_unique_values = sorted(
    valid_pair[
        FEATURE_2
    ]
    .unique()
    .tolist()
)


# ============================================================
# 11. VALIDATE EXPECTED CATEGORIES
# ============================================================

feature_1_invalid_mask = (
    ~valid_pair[
        FEATURE_1
    ]
    .isin(
        VALID_FEATURE_1_VALUES
    )
)


feature_2_invalid_mask = (
    ~valid_pair[
        FEATURE_2
    ]
    .isin(
        VALID_FEATURE_2_VALUES
    )
)


feature_1_invalid_values = int(
    feature_1_invalid_mask.sum()
)


feature_2_invalid_values = int(
    feature_2_invalid_mask.sum()
)


feature_1_invalid_percentage = (
    feature_1_invalid_values
    / valid_observations
    * 100
)


feature_2_invalid_percentage = (
    feature_2_invalid_values
    / valid_observations
    * 100
)


# ============================================================
# 12. KEEP ONLY VALID CATEGORY PAIRS
# ============================================================

valid_binary_pair_mask = (
    (~feature_1_invalid_mask)
    &
    (~feature_2_invalid_mask)
)


binary_pair = (
    valid_pair.loc[
        valid_binary_pair_mask,
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .copy()
)


binary_pair_observations = int(
    len(
        binary_pair
    )
)


if binary_pair_observations == 0:

    raise ValueError(
        "No valid binary-category pairs are available."
    )


excluded_invalid_pairs = (
    valid_observations
    - binary_pair_observations
)


excluded_invalid_pair_percentage = (
    excluded_invalid_pairs
    / valid_observations
    * 100
)


# ============================================================
# 13. INTERNAL NUMERIC BINARY REPRESENTATION
#
# Used only for statistical calculations.
#
# The original dataset is not modified.
#
# SEND_GENDER_BE:
# F -> 0
# M -> 1
#
# TRANS_YEAR_BE:
# 2019 -> 0
# 2020 -> 1
# ============================================================

feature_1_numeric = (
    binary_pair[
        FEATURE_1
    ]
    .map(
        FEATURE_1_BINARY_MAPPING
    )
    .astype(
        "int8"
    )
)


feature_2_numeric = (
    binary_pair[
        FEATURE_2
    ]
    .map(
        FEATURE_2_BINARY_MAPPING
    )
    .astype(
        "int8"
    )
)


# ============================================================
# 14. INDIVIDUAL FEATURE DISTRIBUTIONS
# ============================================================

feature_1_counts = (
    binary_pair[
        FEATURE_1
    ]
    .value_counts()
    .reindex(
        VALID_FEATURE_1_VALUES,
        fill_value=0
    )
)


feature_2_counts = (
    binary_pair[
        FEATURE_2
    ]
    .value_counts()
    .reindex(
        VALID_FEATURE_2_VALUES,
        fill_value=0
    )
)


feature_1_distribution = pd.DataFrame({

    "VALUE":
        VALID_FEATURE_1_VALUES,

    "COUNT":
        [
            int(
                feature_1_counts.loc[
                    value
                ]
            )
            for value
            in VALID_FEATURE_1_VALUES
        ]
})


feature_1_distribution[
    "PERCENTAGE"
] = (
    feature_1_distribution[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


feature_2_distribution = pd.DataFrame({

    "VALUE":
        VALID_FEATURE_2_VALUES,

    "COUNT":
        [
            int(
                feature_2_counts.loc[
                    value
                ]
            )
            for value
            in VALID_FEATURE_2_VALUES
        ]
})


feature_2_distribution[
    "PERCENTAGE"
] = (
    feature_2_distribution[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


# ============================================================
# 15. CONTINGENCY TABLE
# ============================================================

contingency_table = (
    pd.crosstab(
        binary_pair[
            FEATURE_1
        ],
        binary_pair[
            FEATURE_2
        ]
    )
    .reindex(
        index=VALID_FEATURE_1_VALUES,
        columns=VALID_FEATURE_2_VALUES,
        fill_value=0
    )
)


contingency_table.index.name = (
    FEATURE_1
)


contingency_table.columns.name = (
    FEATURE_2
)


contingency_values = (
    contingency_table
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 16. JOINT PERCENTAGE TABLE
# ============================================================

joint_percentage_table = (
    contingency_table
    / binary_pair_observations
    * 100
)


# ============================================================
# 17. CONDITIONAL DISTRIBUTION
#
# TRANS_YEAR_BE given SEND_GENDER_BE.
#
# Each row sums to 100%.
# ============================================================

feature_2_given_feature_1 = (
    contingency_table
    .div(
        contingency_table.sum(
            axis=1
        ),
        axis=0
    )
    * 100
)


feature_2_given_feature_1 = (
    feature_2_given_feature_1
    .fillna(
        0
    )
)


# ============================================================
# 18. CONDITIONAL DISTRIBUTION
#
# SEND_GENDER_BE given TRANS_YEAR_BE.
#
# Each column sums to 100%.
# ============================================================

feature_1_given_feature_2 = (
    contingency_table
    .div(
        contingency_table.sum(
            axis=0
        ),
        axis=1
    )
    * 100
)


feature_1_given_feature_2 = (
    feature_1_given_feature_2
    .fillna(
        0
    )
)


# ============================================================
# 19. JOINT COMBINATION TABLE
# ============================================================

joint_combination_table = (
    contingency_table
    .stack()
    .rename(
        "COUNT"
    )
    .reset_index()
)


joint_combination_table[
    "PERCENTAGE"
] = (
    joint_combination_table[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


# ============================================================
# 20. PHI COEFFICIENT
#
# Phi measures direction and strength of association
# between two dichotomous variables.
#
# For binary variables:
#
# Phi = Pearson correlation.
# ============================================================

phi_coefficient = float(
    np.corrcoef(
        feature_1_numeric,
        feature_2_numeric
    )[
        0,
        1
    ]
)


absolute_phi = abs(
    phi_coefficient
)


# ============================================================
# 21. PHI DIRECTION
# ============================================================

if phi_coefficient > 0:

    phi_direction = (
        "Positive"
    )


elif phi_coefficient < 0:

    phi_direction = (
        "Negative"
    )


else:

    phi_direction = (
        "No directional association"
    )


# ============================================================
# 22. PHI STRENGTH
# ============================================================

if absolute_phi < 0.10:

    phi_strength = (
        "Very weak or negligible"
    )


elif absolute_phi < 0.30:

    phi_strength = (
        "Weak"
    )


elif absolute_phi < 0.50:

    phi_strength = (
        "Moderate"
    )


elif absolute_phi < 0.70:

    phi_strength = (
        "Strong"
    )


else:

    phi_strength = (
        "Very strong"
    )


# ============================================================
# 23. PHI INTERPRETATION
# ============================================================

if phi_direction == "No directional association":

    phi_interpretation = (
        f"Phi = {phi_coefficient:.6f}, indicating "
        f"{phi_strength.lower()} association between "
        f"{FEATURE_1} and {FEATURE_2}, with no directional "
        f"association detected."
    )


else:

    phi_interpretation = (
        f"Phi = {phi_coefficient:.6f}, indicating a "
        f"{phi_strength.lower()} and "
        f"{phi_direction.lower()} association between "
        f"{FEATURE_1} and {FEATURE_2}."
    )


# ============================================================
# 24. CHI-SQUARE TEST OF INDEPENDENCE
#
# H0:
# SEND_GENDER_BE and TRANS_YEAR_BE are independent.
#
# H1:
# SEND_GENDER_BE and TRANS_YEAR_BE are associated.
#
# Degrees of freedom:
#
# df = (rows - 1) * (columns - 1)
#
# df = (2 - 1) * (2 - 1)
#
# df = 1
# ============================================================

(
    chi_square_statistic,
    chi_square_p_value,
    chi_square_degrees_of_freedom,
    expected_frequencies
) = stats.chi2_contingency(
    contingency_values,
    correction=False
)


chi_square_statistic = float(
    chi_square_statistic
)


chi_square_p_value = float(
    chi_square_p_value
)


chi_square_degrees_of_freedom = int(
    chi_square_degrees_of_freedom
)


# ============================================================
# 25. CHI-SQUARE DECISION
# ============================================================

if chi_square_p_value < ALPHA:

    chi_square_decision = (
        "Reject H0"
    )


    chi_square_interpretation = (
        "The Chi-square test provides statistical evidence "
        f"of association between {FEATURE_1} and {FEATURE_2}."
    )


else:

    chi_square_decision = (
        "Fail to reject H0"
    )


    chi_square_interpretation = (
        "The Chi-square test does not provide sufficient "
        f"statistical evidence of association between "
        f"{FEATURE_1} and {FEATURE_2}."
    )


# ============================================================
# 26. CRAMER'S V
#
# Measures association strength between categorical variables.
#
# For a 2 x 2 contingency table:
#
# Cramer's V = |Phi|
# ============================================================

number_rows = int(
    contingency_table.shape[
        0
    ]
)


number_columns = int(
    contingency_table.shape[
        1
    ]
)


minimum_dimension = min(
    number_rows - 1,
    number_columns - 1
)


cramers_v = float(
    np.sqrt(
        chi_square_statistic
        /
        (
            binary_pair_observations
            * minimum_dimension
        )
    )
)


# ============================================================
# 27. CRAMER'S V STRENGTH
# ============================================================

if cramers_v < 0.10:

    cramers_v_strength = (
        "Very weak or negligible"
    )


elif cramers_v < 0.30:

    cramers_v_strength = (
        "Weak"
    )


elif cramers_v < 0.50:

    cramers_v_strength = (
        "Moderate"
    )


elif cramers_v < 0.70:

    cramers_v_strength = (
        "Strong"
    )


else:

    cramers_v_strength = (
        "Very strong"
    )


# ============================================================
# 28. CRAMER'S V INTERPRETATION
# ============================================================

cramers_v_interpretation = (
    f"Cramer's V = {cramers_v:.6f}, indicating a "
    f"{cramers_v_strength.lower()} association between "
    f"{FEATURE_1} and {FEATURE_2}."
)


# ============================================================
# 29. PHI / CRAMER'S V EQUIVALENCE
# ============================================================

phi_cramers_difference = float(
    abs(
        absolute_phi
        - cramers_v
    )
)


# ============================================================
# 30. ASSOCIATION SUMMARY TABLE
# ============================================================

association_summary_table = pd.DataFrame({

    "MEASURE": [
        "Phi coefficient",
        "Absolute Phi",
        "Phi direction",
        "Phi strength",
        "Chi-square statistic",
        "Chi-square p-value",
        "Chi-square degrees of freedom",
        "Chi-square decision",
        "Cramer's V",
        "Cramer's V strength",
        "Absolute Phi - Cramer's V difference"
    ],

    "VALUE": [
        f"{phi_coefficient:.12f}",
        f"{absolute_phi:.12f}",
        phi_direction,
        phi_strength,
        f"{chi_square_statistic:.12f}",
        f"{chi_square_p_value:.12e}",
        str(
            chi_square_degrees_of_freedom
        ),
        chi_square_decision,
        f"{cramers_v:.12f}",
        cramers_v_strength,
        f"{phi_cramers_difference:.12e}"
    ]
})


# ============================================================
# 31. CREATE GROUPED CONDITIONAL PROPORTION CHART
# ============================================================

conditional_values = (
    feature_2_given_feature_1
    .to_numpy(
        dtype="float64"
    )
)


x_positions = np.arange(
    len(
        VALID_FEATURE_1_VALUES
    )
)


bar_width = 0.35


fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


ax.bar(
    x_positions
    - bar_width / 2,
    conditional_values[
        :,
        0
    ],
    width=bar_width,
    label=f"{FEATURE_2} = 2019"
)


ax.bar(
    x_positions
    + bar_width / 2,
    conditional_values[
        :,
        1
    ],
    width=bar_width,
    label=f"{FEATURE_2} = 2020"
)


ax.set_title(
    f"Conditional distribution of {FEATURE_2} by {FEATURE_1}"
)


ax.set_xlabel(
    FEATURE_1
)


ax.set_ylabel(
    "Percentage within group"
)


ax.set_xticks(
    x_positions
)


ax.set_xticklabels(
    VALID_FEATURE_1_VALUES
)


ax.set_ylim(
    0,
    100
)


ax.grid(
    axis="y",
    alpha=0.3
)


ax.legend()


fig.tight_layout()


fig.savefig(
    GROUPED_PROPORTIONS_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 32. FUNCTION TO CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 33. PREPARE TABLES FOR HTML
# ============================================================

feature_1_distribution_html = (
    feature_1_distribution
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


feature_2_distribution_html = (
    feature_2_distribution
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


contingency_table_html = (
    contingency_table
    .to_html(
        border=0
    )
)


joint_percentage_table_html = (
    joint_percentage_table
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


feature_2_given_feature_1_html = (
    feature_2_given_feature_1
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


feature_1_given_feature_2_html = (
    feature_1_given_feature_2
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


joint_combination_table_html = (
    joint_combination_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


association_summary_html = (
    association_summary_table
    .to_html(
        index=False,
        border=0
    )
)


# ============================================================
# 34. FORMAT OBSERVED VALUES
# ============================================================

feature_1_unique_values_text = (
    ", ".join(
        str(
            value
        )
        for value
        in feature_1_unique_values
    )
)


feature_2_unique_values_text = (
    ", ".join(
        str(
            value
        )
        for value
        in feature_2_unique_values
    )
)


# ============================================================
# 35. CONVERT CHART TO BASE64
# ============================================================

grouped_proportions_base64 = (
    image_to_base64(
        GROUPED_PROPORTIONS_PATH
    )
)


# ============================================================
# 36. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Binary Encoding
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis — Binary Encoding
</h1>


<p>

This report evaluates the joint relationship
between:

</p>


<p class="result">

{FEATURE_1}

<br><br>

{FEATURE_2}

</p>


<!-- ========================================================
     1. FEATURE PAIR OVERVIEW
========================================================= -->


<h2>
1. Binary feature pair overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>{FEATURE_1} missing values</td>
<td>{feature_1_missing}</td>
</tr>

<tr>
<td>{FEATURE_2} missing values</td>
<td>{feature_2_missing}</td>
</tr>

<tr>
<td>Rows with at least one missing value</td>
<td>{pair_missing}</td>
</tr>

<tr>
<td>Pair-missing percentage</td>
<td>{pair_missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid pairs analyzed</td>
<td>{binary_pair_observations}</td>
</tr>

</table>


<!-- ========================================================
     2. CATEGORY VALIDATION
========================================================= -->


<h2>
2. Category validation
</h2>


<table>

<tr>
<th>Feature</th>
<th>Expected values</th>
<th>Observed values</th>
<th>Invalid observations</th>
<th>Invalid percentage</th>
</tr>

<tr>
<td>{FEATURE_1}</td>
<td>F, M</td>
<td>{feature_1_unique_values_text}</td>
<td>{feature_1_invalid_values}</td>
<td>{feature_1_invalid_percentage:.6f}%</td>
</tr>

<tr>
<td>{FEATURE_2}</td>
<td>2019, 2020</td>
<td>{feature_2_unique_values_text}</td>
<td>{feature_2_invalid_values}</td>
<td>{feature_2_invalid_percentage:.6f}%</td>
</tr>

</table>


<p>

<strong>Pairs excluded because of invalid categories:</strong>
{excluded_invalid_pairs}

<br>

<strong>Excluded percentage:</strong>
{excluded_invalid_pair_percentage:.6f}%

</p>


<div class="note">

The original dataset is not modified.

For statistical calculations requiring
numerical binary values, the following
internal representation is used:

<br><br>

F = 0

<br>

M = 1

<br><br>

2019 = 0

<br>

2020 = 1

</div>


<!-- ========================================================
     3. INDIVIDUAL DISTRIBUTIONS
========================================================= -->


<h2>
3. Individual distributions
</h2>


<h3>
{FEATURE_1}
</h3>


{feature_1_distribution_html}


<h3>
{FEATURE_2}
</h3>


{feature_2_distribution_html}


<!-- ========================================================
     4. JOINT FREQUENCY TABLE
========================================================= -->


<h2>
4. Joint frequency table
</h2>


<p>

The contingency table shows the observed number
of transactions for each combination of
{FEATURE_1} and {FEATURE_2}.

</p>


{contingency_table_html}


<h3>
Combination-level representation
</h3>


{joint_combination_table_html}


<!-- ========================================================
     5. JOINT PERCENTAGE TABLE
========================================================= -->


<h2>
5. Joint percentage table
</h2>


<p>

Each cell represents its percentage relative
to all valid feature pairs.

</p>


{joint_percentage_table_html}


<!-- ========================================================
     6. CONDITIONAL DISTRIBUTIONS
========================================================= -->


<h2>
6. Conditional distributions
</h2>


<h3>
{FEATURE_2} given {FEATURE_1}
</h3>


<p>

Each row sums to approximately 100%.

This table shows how transaction years are
distributed within each gender category.

</p>


{feature_2_given_feature_1_html}


<h3>
{FEATURE_1} given {FEATURE_2}
</h3>


<p>

Each column sums to approximately 100%.

This table shows how gender categories are
distributed within each transaction year.

</p>


{feature_1_given_feature_2_html}


<!-- ========================================================
     7. PHI COEFFICIENT
========================================================= -->


<h2>
7. Phi coefficient
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Phi coefficient</td>
<td>{phi_coefficient:.12f}</td>
</tr>

<tr>
<td>Absolute Phi</td>
<td>{absolute_phi:.12f}</td>
</tr>

<tr>
<td>Direction</td>
<td>{phi_direction}</td>
</tr>

<tr>
<td>Descriptive strength</td>
<td>{phi_strength}</td>
</tr>

</table>


<p class="result">

{phi_interpretation}

</p>


<div class="note">

Phi measures both the direction and the
strength of association between two
dichotomous variables.

<br><br>

For this analysis:

<br><br>

F = 0 and M = 1

<br>

2019 = 0 and 2020 = 1

<br><br>

Therefore, a positive Phi indicates a tendency
for equal binary codes to occur together,
whereas a negative Phi indicates a tendency
for opposite binary codes to occur together.

<br><br>

The sign depends on the coding orientation.
The magnitude describes the strength of
association.

</div>


<!-- ========================================================
     8. CHI-SQUARE TEST
========================================================= -->


<h2>
8. Chi-square test of independence
</h2>


<p>

<strong>H0:</strong>
{FEATURE_1} and {FEATURE_2} are independent.

<br><br>

<strong>H1:</strong>
{FEATURE_1} and {FEATURE_2} are associated.

<br><br>

<strong>Significance level:</strong>
α = {ALPHA}

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Chi-square statistic</td>
<td>{chi_square_statistic:.12f}</td>
</tr>

<tr>
<td>Degrees of freedom</td>
<td>{chi_square_degrees_of_freedom}</td>
</tr>

<tr>
<td>p-value</td>
<td>{chi_square_p_value:.12e}</td>
</tr>

<tr>
<td>Decision</td>
<td>{chi_square_decision}</td>
</tr>

</table>


<p class="result">

{chi_square_interpretation}

</p>


<div class="note">

<strong>Why is there 1 degree of freedom?</strong>

<br><br>

The contingency table has 2 rows
(F and M) and 2 columns
(2019 and 2020).

<br><br>

For a Chi-square test of independence:

<br><br>

df = (rows - 1) × (columns - 1)

<br><br>

df = (2 - 1) × (2 - 1)

<br><br>

<strong>df = 1</strong>

<br><br>

Once the row totals and column totals are fixed,
only one cell can vary freely.
The remaining three cell frequencies are then
mathematically determined.

</div>


<!-- ========================================================
     9. CRAMER'S V
========================================================= -->


<h2>
9. Cramer's V
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Cramer's V</td>
<td>{cramers_v:.12f}</td>
</tr>

<tr>
<td>Association strength</td>
<td>{cramers_v_strength}</td>
</tr>

<tr>
<td>Absolute Phi</td>
<td>{absolute_phi:.12f}</td>
</tr>

<tr>
<td>Numerical difference</td>
<td>{phi_cramers_difference:.12e}</td>
</tr>

</table>


<p class="result">

{cramers_v_interpretation}

</p>


<div class="note">

Cramer's V measures the magnitude of association
between categorical variables.

It ranges from 0 to 1 and does not indicate
association direction.

<br><br>

For a 2 × 2 contingency table,
Cramer's V is equal to the absolute value of Phi:

<br><br>

<strong>V = |Phi|</strong>

<br><br>

Phi retains the direction of association,
whereas Cramer's V expresses association
magnitude only.

</div>


<!-- ========================================================
     10. GROUPED CONDITIONAL PROPORTIONS
========================================================= -->


<h2>
10. Grouped conditional proportions
</h2>


<p>

The chart compares the percentage distribution
of transaction year within each gender category.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{grouped_proportions_base64}"
    alt="Grouped conditional proportions"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


{association_summary_html}


<div class="note">

<strong>Combined interpretation:</strong>

<br><br>

The Chi-square test evaluates whether statistical
evidence of association exists.

<br><br>

Phi evaluates both the direction and the magnitude
of the binary-binary association.

<br><br>

Cramer's V evaluates the magnitude of the
categorical association without considering
direction.

<br><br>

Because the dataset contains a very large number
of observations, statistical significance can
occur even when the practical magnitude of the
association is small.

Therefore, the Chi-square p-value should be
interpreted together with Phi, Cramer's V,
the contingency table and the conditional
proportions.

</div>


</body>

</html>
"""


# ============================================================
# 37. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 38. DISPLAY ANALYSIS OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "BINARY ENCODING - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "Feature 1:",
    FEATURE_1
)


print(
    "Observed values:",
    feature_1_unique_values
)


print(
    "\nFeature 2:",
    FEATURE_2
)


print(
    "Observed values:",
    feature_2_unique_values
)


print(
    "\nValid pairs:",
    binary_pair_observations
)


# ============================================================
# 39. DISPLAY INDIVIDUAL DISTRIBUTIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_1} DISTRIBUTION"
)


print(
    "=" * 100
)


display(
    feature_1_distribution
)


print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_2} DISTRIBUTION"
)


print(
    "=" * 100
)


display(
    feature_2_distribution
)


# ============================================================
# 40. DISPLAY CONTINGENCY TABLE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT FREQUENCY TABLE"
)


print(
    "=" * 100
)


display(
    contingency_table
)


# ============================================================
# 41. DISPLAY JOINT PERCENTAGE TABLE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT PERCENTAGE TABLE"
)


print(
    "=" * 100
)


display(
    joint_percentage_table
)


# ============================================================
# 42. DISPLAY CONDITIONAL DISTRIBUTIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_2} GIVEN {FEATURE_1}"
)


print(
    "=" * 100
)


display(
    feature_2_given_feature_1
)


print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_1} GIVEN {FEATURE_2}"
)


print(
    "=" * 100
)


display(
    feature_1_given_feature_2
)


# ============================================================
# 43. DISPLAY PHI
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PHI COEFFICIENT"
)


print(
    "=" * 100
)


print(
    "Phi coefficient:",
    f"{phi_coefficient:.12f}"
)


print(
    "Absolute Phi:",
    f"{absolute_phi:.12f}"
)


print(
    "Direction:",
    phi_direction
)


print(
    "Strength:",
    phi_strength
)


print(
    "Interpretation:",
    phi_interpretation
)


# ============================================================
# 44. DISPLAY CHI-SQUARE TEST
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CHI-SQUARE TEST OF INDEPENDENCE"
)


print(
    "=" * 100
)


print(
    "Statistic:",
    f"{chi_square_statistic:.12f}"
)


print(
    "Degrees of freedom:",
    chi_square_degrees_of_freedom
)


print(
    "p-value:",
    f"{chi_square_p_value:.12e}"
)


print(
    "Decision:",
    chi_square_decision
)


print(
    "Interpretation:",
    chi_square_interpretation
)


print(
    "\nDegrees of freedom formula:"
)


print(
    "(2 - 1) * (2 - 1) = 1"
)


# ============================================================
# 45. DISPLAY CRAMER'S V
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CRAMER'S V"
)


print(
    "=" * 100
)


print(
    "Cramer's V:",
    f"{cramers_v:.12f}"
)


print(
    "Strength:",
    cramers_v_strength
)


print(
    "Interpretation:",
    cramers_v_interpretation
)


print(
    "Absolute Phi:",
    f"{absolute_phi:.12f}"
)


print(
    "Difference between |Phi| and Cramer's V:",
    f"{phi_cramers_difference:.12e}"
)


# ============================================================
# 46. DISPLAY ASSOCIATION SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ASSOCIATION SUMMARY"
)


print(
    "=" * 100
)


display(
    association_summary_table
)


# ============================================================
# 47. RELEASE MEMORY
# ============================================================

del dataset_features
del valid_pair
del binary_pair

del feature_1_numeric
del feature_2_numeric

del contingency_values
del expected_frequencies
del conditional_values

gc.collect()


# ============================================================
# 48. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nHTML:"
)


print(
    HTML_PATH
)


print(
    "\nPNG:"
)


print(
    GROUPED_PROPORTIONS_PATH
)


BINARY ENCODING - JOINT ANALYSIS
Feature 1: SEND_GENDER_BE
Observed values: ['F', 'M']

Feature 2: TRANS_YEAR_BE
Observed values: [2019, 2020]

Valid pairs: 1852394

SEND_GENDER_BE DISTRIBUTION


,VALUE,COUNT,PERCENTAGE
0,F,1014749,54.780408
1,M,837645,45.219592



TRANS_YEAR_BE DISTRIBUTION


,VALUE,COUNT,PERCENTAGE
0,2019,924850,49.927283
1,2020,927544,50.072717



JOINT FREQUENCY TABLE


...


SEND_GENDER_BE GIVEN TRANS_YEAR_BE


TRANS_YEAR_BE,2019,2020
SEND_GENDER_BE,,
F,54.724226,54.836428
M,45.275774,45.163572



PHI COEFFICIENT
Phi coefficient: -0.001127189364
Absolute Phi: 0.001127189364
Direction: Negative
Strength: Very weak or negligible
Interpretation: Phi = -0.001127, indicating a very weak or negligible and negative association between SEND_GENDER_BE and TRANS_YEAR_BE.

CHI-SQUARE TEST OF INDEPENDENCE
Statistic: 2.353570055418
Degrees of freedom: 1
p-value: 1.249964558622e-01
Decision: Fail to reject H0
Interpretation: The Chi-square test does not provide sufficient statistical evidence of association between SEND_GENDER_BE and TRANS_YEAR_BE.

Degrees of freedom formula:
(2 - 1) * (2 - 1) = 1

CRAMER'S V
Cramer's V: 0.001127189364
Strength: Very weak or negligible
Interpretation: Cramer's V = 0.001127, indicating a very weak or negligible association between SEND_GENDER_BE and TRANS_YEAR_BE.
Absolute Phi: 0.001127189364
Difference between |Phi| and Cramer's V: 2.144551897176e-16

ASSOCIATION SUMMARY


,MEASURE,VALUE
0,Phi coefficient,-0.001127189364
1,Absolute Phi,0.001127189364
2,Phi direction,Negative
3,Phi strength,Very weak or negligible
4,Chi-square statistic,2.353570055418
5,Chi-square p-value,1.249964558622e-01
6,Chi-square degrees of freedom,1
7,Chi-square decision,Fail to reject H0
8,Cramer's V,0.001127189364
9,Cramer's V strength,Very weak or negligible



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding/analysis_binary_encoding.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding/binary_encoding_grouped_proportions.png


## <span style="color:PURPLE"> CONTINUOS GEOGRAPHIC </span> ##

In [7]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "continuous_geographic"
)

SEND_LAT = (
    "SEND_LAT_REGISTER"
)

SEND_LONG = (
    "SEND_LONG_REGISTER"
)

RECEIVE_LAT = (
    "RECEIVE_LAT"
)

RECEIVE_LONG = (
    "RECEIVE_LONG"
)

GEOGRAPHIC_FEATURES = [
    SEND_LAT,
    SEND_LONG,
    RECEIVE_LAT,
    RECEIVE_LONG
]


# ============================================================
# 02. GENERAL SETTINGS
# ============================================================

EARTH_RADIUS_KM = 6371.0088

RANDOM_STATE = 42

HIGH_CORRELATION_THRESHOLD = 0.90

ALPHA = 0.05

QQ_SAMPLE_MAXIMUM = 50000

PNG_DPI = 300


# ============================================================
# 03. MAP SETTINGS
# ============================================================

MAP_SAMPLE_FRACTION = 0.01

DIRECTION_MAP_SAMPLE_FRACTION = 0.001

DIRECTION_MAP_RANDOM_STATE = 42

ARROW_HEAD_ANGLE_DEGREES = 28

ARROW_HEAD_MINIMUM_SIZE = 0.015

ARROW_HEAD_MAXIMUM_SIZE = 0.080

ARROW_HEAD_RELATIVE_SIZE = 0.20


# ============================================================
# 04. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 05. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 06. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_continuous_geographic.html"
)


WORLD_MAP_HTML_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_map.html"
)


WORLD_MAP_PNG_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_map.png"
)


DIRECTION_MAP_HTML_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_direction_map.html"
)


DIRECTION_MAP_PNG_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_direction_map.png"
)


PEARSON_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_pearson_correlation.png"
)


SPEARMAN_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_spearman_correlation.png"
)


DISTANCE_DISTRIBUTION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_distribution.png"
)


DISTANCE_QQ_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_qqplot.png"
)


# ============================================================
# 08. REMOVE OBSOLETE OUTPUT FILES
# ============================================================

OBSOLETE_OUTPUTS = [

    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_scatter.png",

    RESULTS_DIRECTORY
    / "continuous_geographic_latitude_relationship.png",

    RESULTS_DIRECTORY
    / "continuous_geographic_longitude_relationship.png"
]


for obsolete_output in OBSOLETE_OUTPUTS:

    if obsolete_output.exists():

        obsolete_output.unlink()


for optional_png in [
    WORLD_MAP_PNG_PATH,
    DIRECTION_MAP_PNG_PATH
]:

    if optional_png.exists():

        optional_png.unlink()


# ============================================================
# 09. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 10. LOAD ONLY GEOGRAPHIC FEATURES
# ============================================================

dataset_geo = pd.read_parquet(
    DATASET_PATH,
    columns=GEOGRAPHIC_FEATURES
)


total_observations = int(
    len(
        dataset_geo
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 11. KEEP COMPLETE FINITE OBSERVATIONS
#
# All statistical analyses use these observations.
#
# The original dataset is not modified.
# ============================================================

complete_geo = (
    dataset_geo[
        GEOGRAPHIC_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_geo[
        GEOGRAPHIC_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_geo = (
    complete_geo.loc[
        finite_mask,
        GEOGRAPHIC_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_geo
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite geographic observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 12. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_statistics = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)


descriptive_statistics[
    "variance"
] = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .var()
)


descriptive_statistics[
    "range"
] = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .max()
    -
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .min()
)


# ============================================================
# 13. RANDOM 1% SAMPLE FOR THE FIRST MAP
#
# This sample is used only for visualization.
# ============================================================

map_sample_size = max(
    1,
    int(
        round(
            total_observations
            * MAP_SAMPLE_FRACTION
        )
    )
)


map_sample = (
    dataset_geo
    .sample(
        n=map_sample_size,
        random_state=RANDOM_STATE
    )
    .copy()
)


# ============================================================
# 14. PREPARE SENDER SAMPLE FOR FIRST MAP
# ============================================================

sender_map_sample = (
    map_sample[
        [
            SEND_LAT,
            SEND_LONG
        ]
    ]
    .dropna()
    .copy()
)


sender_map_finite_mask = np.isfinite(
    sender_map_sample[
        [
            SEND_LAT,
            SEND_LONG
        ]
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


sender_map_sample = (
    sender_map_sample.loc[
        sender_map_finite_mask
    ]
    .copy()
)


# ============================================================
# 15. PREPARE RECEIVER SAMPLE FOR FIRST MAP
# ============================================================

receiver_map_sample = (
    map_sample[
        [
            RECEIVE_LAT,
            RECEIVE_LONG
        ]
    ]
    .dropna()
    .copy()
)


receiver_map_finite_mask = np.isfinite(
    receiver_map_sample[
        [
            RECEIVE_LAT,
            RECEIVE_LONG
        ]
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


receiver_map_sample = (
    receiver_map_sample.loc[
        receiver_map_finite_mask
    ]
    .copy()
)


sender_map_observations = int(
    len(
        sender_map_sample
    )
)


receiver_map_observations = int(
    len(
        receiver_map_sample
    )
)


# ============================================================
# 16. CREATE FIRST INTERACTIVE MAP
#
# BLUE:
# Sender registered locations
#
# RED:
# Receiver locations
#
# Random 1% sample.
# ============================================================

world_map_figure = go.Figure()


world_map_figure.add_trace(

    go.Scattergeo(

        lon=sender_map_sample[
            SEND_LONG
        ],

        lat=sender_map_sample[
            SEND_LAT
        ],

        mode="markers",

        name="Sender registered location",

        marker=dict(
            size=4,
            color="#0000FF",
            opacity=0.90
        ),

        hovertemplate=(
            "<b>Sender registered location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


world_map_figure.add_trace(

    go.Scattergeo(

        lon=receiver_map_sample[
            RECEIVE_LONG
        ],

        lat=receiver_map_sample[
            RECEIVE_LAT
        ],

        mode="markers",

        name="Receiver location",

        marker=dict(
            size=4,
            color="#FF0000",
            opacity=0.90
        ),

        hovertemplate=(
            "<b>Receiver location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


world_map_figure.update_geos(

    projection_type="natural earth",

    showland=True,

    landcolor="rgb(238,238,238)",

    showocean=True,

    oceancolor="rgb(245,248,250)",

    showcountries=True,

    countrycolor="rgb(160,160,160)",

    showcoastlines=True,

    coastlinecolor="rgb(100,100,100)"
)


world_map_figure.update_layout(

    title=(
        "Sender and receiver geographic locations "
        "- random 1% sample"
    ),

    width=1400,

    height=800,

    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)


# ============================================================
# 17. SAVE FIRST INTERACTIVE MAP
# ============================================================

world_map_figure.write_html(

    str(
        WORLD_MAP_HTML_PATH
    ),

    include_plotlyjs=True,

    full_html=True,

    config={
        "responsive": True,
        "displaylogo": False
    }
)


# ============================================================
# 18. CREATE FIRST MAP HTML FRAGMENT
# ============================================================

world_map_html_fragment = (
    world_map_figure
    .to_html(
        full_html=False,
        include_plotlyjs=True,
        config={
            "responsive": True,
            "displaylogo": False
        }
    )
)


# ============================================================
# 19. ATTEMPT FIRST MAP STATIC EXPORT
# ============================================================

map_png_created = False

map_png_error = ""


try:

    world_map_figure.write_image(

        str(
            WORLD_MAP_PNG_PATH
        ),

        width=1600,

        height=900,

        scale=2
    )


    map_png_created = True


except Exception as error:

    map_png_error = str(
        error
    )


# ============================================================
# 20. RANDOM 0.1% SAMPLE FOR DIRECTION MAP
#
# Sender and receiver coordinates remain in the
# same row because each arrow represents one transaction.
# ============================================================

direction_map_sample_size = max(
    1,
    int(
        round(
            total_observations
            * DIRECTION_MAP_SAMPLE_FRACTION
        )
    )
)


direction_map_sample = (
    dataset_geo
    .sample(
        n=direction_map_sample_size,
        random_state=DIRECTION_MAP_RANDOM_STATE
    )
    .dropna(
        subset=GEOGRAPHIC_FEATURES
    )
    .copy()
)


# ============================================================
# 21. KEEP FINITE COORDINATES FOR DIRECTION MAP
# ============================================================

direction_finite_mask = np.isfinite(
    direction_map_sample[
        GEOGRAPHIC_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


direction_map_sample = (
    direction_map_sample.loc[
        direction_finite_mask
    ]
    .copy()
)


direction_map_observations = int(
    len(
        direction_map_sample
    )
)


# ============================================================
# 22. PREPARE DIRECTION MAP ARRAYS
# ============================================================

direction_send_lat = (
    direction_map_sample[
        SEND_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_send_long = (
    direction_map_sample[
        SEND_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_receive_lat = (
    direction_map_sample[
        RECEIVE_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_receive_long = (
    direction_map_sample[
        RECEIVE_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 23. CREATE GREEN DIRECTIONAL ARROWS
#
# Each arrow starts at the sender location and ends
# at the receiver location.
#
# The line and arrowhead are combined into a single
# Plotly trace for better performance.
# ============================================================

arrow_longitudes = []

arrow_latitudes = []

direction_arrow_count = 0


arrow_head_angle = np.radians(
    ARROW_HEAD_ANGLE_DEGREES
)


for (
    sender_latitude,
    sender_longitude,
    receiver_latitude,
    receiver_longitude
) in zip(
    direction_send_lat,
    direction_send_long,
    direction_receive_lat,
    direction_receive_long
):

    # --------------------------------------------------------
    # Local correction for longitude scale
    # --------------------------------------------------------

    mean_latitude_rad = np.radians(
        (
            sender_latitude
            + receiver_latitude
        )
        / 2
    )


    longitude_scale = np.cos(
        mean_latitude_rad
    )


    if abs(
        longitude_scale
    ) < 1e-8:

        longitude_scale = 1e-8


    # --------------------------------------------------------
    # Local Cartesian approximation
    # --------------------------------------------------------

    delta_x = (
        (
            receiver_longitude
            - sender_longitude
        )
        * longitude_scale
    )


    delta_y = (
        receiver_latitude
        - sender_latitude
    )


    segment_length = np.sqrt(
        delta_x ** 2
        + delta_y ** 2
    )


    # --------------------------------------------------------
    # Skip identical locations
    # --------------------------------------------------------

    if segment_length < 1e-12:

        continue


    direction_arrow_count += 1


    # --------------------------------------------------------
    # Arrow shaft
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            sender_longitude,
            receiver_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            sender_latitude,
            receiver_latitude,
            None
        ]
    )


    # --------------------------------------------------------
    # Direction angle
    # --------------------------------------------------------

    direction_angle = np.arctan2(
        delta_y,
        delta_x
    )


    # --------------------------------------------------------
    # Adaptive arrowhead size
    # --------------------------------------------------------

    arrow_head_size = np.clip(

        segment_length
        * ARROW_HEAD_RELATIVE_SIZE,

        ARROW_HEAD_MINIMUM_SIZE,

        ARROW_HEAD_MAXIMUM_SIZE
    )


    backward_angle = (
        direction_angle
        + np.pi
    )


    left_angle = (
        backward_angle
        + arrow_head_angle
    )


    right_angle = (
        backward_angle
        - arrow_head_angle
    )


    # --------------------------------------------------------
    # Left arrowhead point
    # --------------------------------------------------------

    left_delta_x = (
        arrow_head_size
        * np.cos(
            left_angle
        )
    )


    left_delta_y = (
        arrow_head_size
        * np.sin(
            left_angle
        )
    )


    left_longitude = (
        receiver_longitude
        + (
            left_delta_x
            / longitude_scale
        )
    )


    left_latitude = (
        receiver_latitude
        + left_delta_y
    )


    # --------------------------------------------------------
    # Right arrowhead point
    # --------------------------------------------------------

    right_delta_x = (
        arrow_head_size
        * np.cos(
            right_angle
        )
    )


    right_delta_y = (
        arrow_head_size
        * np.sin(
            right_angle
        )
    )


    right_longitude = (
        receiver_longitude
        + (
            right_delta_x
            / longitude_scale
        )
    )


    right_latitude = (
        receiver_latitude
        + right_delta_y
    )


    # --------------------------------------------------------
    # Left side of arrowhead
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            receiver_longitude,
            left_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            receiver_latitude,
            left_latitude,
            None
        ]
    )


    # --------------------------------------------------------
    # Right side of arrowhead
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            receiver_longitude,
            right_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            receiver_latitude,
            right_latitude,
            None
        ]
    )


# ============================================================
# 24. CREATE SECOND INTERACTIVE DIRECTION MAP
# ============================================================

direction_map_figure = go.Figure()


# ============================================================
# 25. ADD GREEN ARROWS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=arrow_longitudes,

        lat=arrow_latitudes,

        mode="lines",

        name="Sender → Receiver",

        line=dict(
            color="#00A000",
            width=1.5
        ),

        opacity=0.85,

        hoverinfo="skip"
    )
)


# ============================================================
# 26. ADD BLUE SENDER LOCATIONS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=direction_send_long,

        lat=direction_send_lat,

        mode="markers",

        name="Sender registered location",

        marker=dict(
            size=5,
            color="#0000FF",
            opacity=0.95
        ),

        hovertemplate=(
            "<b>Sender registered location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 27. ADD RED RECEIVER LOCATIONS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=direction_receive_long,

        lat=direction_receive_lat,

        mode="markers",

        name="Receiver location",

        marker=dict(
            size=5,
            color="#FF0000",
            opacity=0.95
        ),

        hovertemplate=(
            "<b>Receiver location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 28. SECOND MAP APPEARANCE
# ============================================================

direction_map_figure.update_geos(

    projection_type="natural earth",

    fitbounds="locations",

    showland=True,

    landcolor="rgb(238,238,238)",

    showocean=True,

    oceancolor="rgb(245,248,250)",

    showcountries=True,

    countrycolor="rgb(150,150,150)",

    showcoastlines=True,

    coastlinecolor="rgb(90,90,90)",

    showlakes=True,

    lakecolor="rgb(245,248,250)"
)


direction_map_figure.update_layout(

    title=(
        "Sender-to-receiver geographic direction "
        "- random 0.1% sample"
    ),

    width=1400,

    height=850,

    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)


# ============================================================
# 29. SAVE SECOND INTERACTIVE MAP
# ============================================================

direction_map_figure.write_html(

    str(
        DIRECTION_MAP_HTML_PATH
    ),

    include_plotlyjs=True,

    full_html=True,

    config={
        "responsive": True,
        "displaylogo": False
    }
)


# ============================================================
# 30. CREATE SECOND MAP HTML FRAGMENT
#
# Plotly JavaScript was already included by the first map.
# ============================================================

direction_map_html_fragment = (
    direction_map_figure
    .to_html(
        full_html=False,
        include_plotlyjs=False,
        config={
            "responsive": True,
            "displaylogo": False
        }
    )
)


# ============================================================
# 31. ATTEMPT SECOND MAP STATIC EXPORT
# ============================================================

direction_map_png_created = False

direction_map_png_error = ""


try:

    direction_map_figure.write_image(

        str(
            DIRECTION_MAP_PNG_PATH
        ),

        width=1600,

        height=950,

        scale=2
    )


    direction_map_png_created = True


except Exception as error:

    direction_map_png_error = str(
        error
    )


# ============================================================
# 32. PEARSON CORRELATION MATRIX
# ============================================================

pearson_matrix = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .corr(
        method="pearson"
    )
)


# ============================================================
# 33. SPEARMAN CORRELATION MATRIX
# ============================================================

spearman_matrix = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


# ============================================================
# 34. CORRELATION INTERPRETATION FUNCTION
# ============================================================

def interpret_correlation(
    value
):

    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        strength = (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        strength = (
            "Weak"
        )


    elif absolute_value < 0.50:

        strength = (
            "Moderate"
        )


    elif absolute_value < 0.70:

        strength = (
            "Strong"
        )


    else:

        strength = (
            "Very strong"
        )


    if value > 0:

        direction = (
            "Positive"
        )


    elif value < 0:

        direction = (
            "Negative"
        )


    else:

        direction = (
            "No directional association"
        )


    return (
        strength,
        direction
    )


# ============================================================
# 35. MAIN SENDER-RECEIVER CORRELATIONS
# ============================================================

latitude_pearson = float(
    pearson_matrix.loc[
        SEND_LAT,
        RECEIVE_LAT
    ]
)


latitude_spearman = float(
    spearman_matrix.loc[
        SEND_LAT,
        RECEIVE_LAT
    ]
)


longitude_pearson = float(
    pearson_matrix.loc[
        SEND_LONG,
        RECEIVE_LONG
    ]
)


longitude_spearman = float(
    spearman_matrix.loc[
        SEND_LONG,
        RECEIVE_LONG
    ]
)


(
    latitude_pearson_strength,
    latitude_pearson_direction
) = interpret_correlation(
    latitude_pearson
)


(
    latitude_spearman_strength,
    latitude_spearman_direction
) = interpret_correlation(
    latitude_spearman
)


(
    longitude_pearson_strength,
    longitude_pearson_direction
) = interpret_correlation(
    longitude_pearson
)


(
    longitude_spearman_strength,
    longitude_spearman_direction
) = interpret_correlation(
    longitude_spearman
)


# ============================================================
# 36. MAIN CORRELATION SUMMARY
# ============================================================

correlation_summary_table = pd.DataFrame({

    "RELATIONSHIP": [
        f"{SEND_LAT} × {RECEIVE_LAT}",
        f"{SEND_LAT} × {RECEIVE_LAT}",
        f"{SEND_LONG} × {RECEIVE_LONG}",
        f"{SEND_LONG} × {RECEIVE_LONG}"
    ],

    "METHOD": [
        "Pearson",
        "Spearman",
        "Pearson",
        "Spearman"
    ],

    "COEFFICIENT": [
        latitude_pearson,
        latitude_spearman,
        longitude_pearson,
        longitude_spearman
    ],

    "DIRECTION": [
        latitude_pearson_direction,
        latitude_spearman_direction,
        longitude_pearson_direction,
        longitude_spearman_direction
    ],

    "STRENGTH": [
        latitude_pearson_strength,
        latitude_spearman_strength,
        longitude_pearson_strength,
        longitude_spearman_strength
    ]
})


# ============================================================
# 37. IDENTIFY HIGH CORRELATIONS
# ============================================================

def identify_high_correlations(
    correlation_matrix,
    method,
    threshold
):

    records = []

    features = (
        correlation_matrix
        .columns
        .tolist()
    )


    for first_index in range(
        len(
            features
        )
    ):

        for second_index in range(
            first_index + 1,
            len(
                features
            )
        ):

            first_feature = (
                features[
                    first_index
                ]
            )


            second_feature = (
                features[
                    second_index
                ]
            )


            correlation_value = float(
                correlation_matrix.loc[
                    first_feature,
                    second_feature
                ]
            )


            if abs(
                correlation_value
            ) >= threshold:

                (
                    strength,
                    direction
                ) = interpret_correlation(
                    correlation_value
                )


                records.append({

                    "METHOD":
                        method,

                    "FEATURE_1":
                        first_feature,

                    "FEATURE_2":
                        second_feature,

                    "CORRELATION":
                        correlation_value,

                    "ABSOLUTE_CORRELATION":
                        abs(
                            correlation_value
                        ),

                    "DIRECTION":
                        direction,

                    "STRENGTH":
                        strength,

                    "POTENTIAL_REDUNDANCY":
                        "Yes"
                })


    return pd.DataFrame(
        records
    )


# ============================================================
# 38. IDENTIFY POTENTIAL REDUNDANCY
# ============================================================

high_pearson_correlations = (
    identify_high_correlations(
        pearson_matrix,
        "Pearson",
        HIGH_CORRELATION_THRESHOLD
    )
)


high_spearman_correlations = (
    identify_high_correlations(
        spearman_matrix,
        "Spearman",
        HIGH_CORRELATION_THRESHOLD
    )
)


high_correlations = pd.concat(
    [
        high_pearson_correlations,
        high_spearman_correlations
    ],
    ignore_index=True
)


number_high_correlations = int(
    len(
        high_correlations
    )
)


# ============================================================
# 39. REDUNDANCY INTERPRETATION
# ============================================================

if number_high_correlations > 0:

    redundancy_interpretation = (
        f"At least one geographic feature pair presents an "
        f"absolute correlation equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"This indicates potential redundancy among the "
        f"geographic features. No feature is removed during "
        f"this exploratory stage. These relationships should "
        f"be considered later in multicollinearity analysis, "
        f"dimensionality reduction and model construction."
    )


else:

    redundancy_interpretation = (
        f"No geographic feature pair presents an absolute "
        f"correlation equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"No strong evidence of geographic redundancy was "
        f"identified using this threshold."
    )


# ============================================================
# 40. CREATE CORRELATION MATRIX PLOT
# ============================================================

def create_correlation_plot(
    correlation_matrix,
    title,
    output_path
):

    values = (
        correlation_matrix
        .to_numpy(
            dtype="float64"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            8
        )
    )


    image = ax.imshow(
        values,
        vmin=-1,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                GEOGRAPHIC_FEATURES
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                GEOGRAPHIC_FEATURES
            )
        )
    )


    ax.set_xticklabels(
        GEOGRAPHIC_FEATURES,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        GEOGRAPHIC_FEATURES
    )


    for row_index in range(
        len(
            GEOGRAPHIC_FEATURES
        )
    ):

        for column_index in range(
            len(
                GEOGRAPHIC_FEATURES
            )
        ):

            ax.text(
                column_index,
                row_index,
                (
                    f"{values[row_index, column_index]:.3f}"
                ),
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label="Correlation coefficient"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 41. CREATE PEARSON CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    pearson_matrix,
    "Pearson correlation matrix - geographic features",
    PEARSON_CORRELATION_PATH
)


# ============================================================
# 42. CREATE SPEARMAN CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    spearman_matrix,
    "Spearman correlation matrix - geographic features",
    SPEARMAN_CORRELATION_PATH
)


# ============================================================
# 43. PREPARE ARRAYS FOR HAVERSINE DISTANCE
# ============================================================

send_latitude = (
    analysis_geo[
        SEND_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


send_longitude = (
    analysis_geo[
        SEND_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


receive_latitude = (
    analysis_geo[
        RECEIVE_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


receive_longitude = (
    analysis_geo[
        RECEIVE_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 44. CALCULATE HAVERSINE DISTANCE
# ============================================================

lat_1_rad = np.radians(
    send_latitude
)


lon_1_rad = np.radians(
    send_longitude
)


lat_2_rad = np.radians(
    receive_latitude
)


lon_2_rad = np.radians(
    receive_longitude
)


delta_latitude = (
    lat_2_rad
    - lat_1_rad
)


delta_longitude = (
    lon_2_rad
    - lon_1_rad
)


haversine_a = (
    np.sin(
        delta_latitude / 2
    ) ** 2
    +
    np.cos(
        lat_1_rad
    )
    *
    np.cos(
        lat_2_rad
    )
    *
    np.sin(
        delta_longitude / 2
    ) ** 2
)


haversine_a = np.clip(
    haversine_a,
    0,
    1
)


central_angle = (
    2
    * np.arctan2(
        np.sqrt(
            haversine_a
        ),
        np.sqrt(
            1
            - haversine_a
        )
    )
)


haversine_distance_km = (
    EARTH_RADIUS_KM
    * central_angle
)


# ============================================================
# 45. HAVERSINE DISTANCE DESCRIPTIVE STATISTICS
# ============================================================

distance_minimum = float(
    np.min(
        haversine_distance_km
    )
)


distance_maximum = float(
    np.max(
        haversine_distance_km
    )
)


distance_mean = float(
    np.mean(
        haversine_distance_km
    )
)


distance_median = float(
    np.median(
        haversine_distance_km
    )
)


distance_standard_deviation = float(
    np.std(
        haversine_distance_km,
        ddof=1
    )
)


distance_variance = float(
    np.var(
        haversine_distance_km,
        ddof=1
    )
)


distance_percentiles = np.percentile(
    haversine_distance_km,
    [
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99
    ]
)


distance_skewness = float(
    stats.skew(
        haversine_distance_km,
        bias=False
    )
)


distance_kurtosis = float(
    stats.kurtosis(
        haversine_distance_km,
        fisher=True,
        bias=False
    )
)


distance_summary_table = pd.DataFrame({

    "METRIC": [
        "Minimum",
        "P1",
        "P5",
        "P10",
        "P25",
        "Median",
        "Mean",
        "P75",
        "P90",
        "P95",
        "P99",
        "Maximum",
        "Standard deviation",
        "Variance",
        "Skewness",
        "Excess kurtosis"
    ],

    "VALUE": [
        distance_minimum,
        distance_percentiles[0],
        distance_percentiles[1],
        distance_percentiles[2],
        distance_percentiles[3],
        distance_median,
        distance_mean,
        distance_percentiles[5],
        distance_percentiles[6],
        distance_percentiles[7],
        distance_percentiles[8],
        distance_maximum,
        distance_standard_deviation,
        distance_variance,
        distance_skewness,
        distance_kurtosis
    ]
})


# ============================================================
# 46. SHAPIRO-WILK NORMALITY TEST
# ============================================================

with warnings.catch_warnings(
    record=True
) as captured_shapiro_warnings:

    warnings.simplefilter(
        "always"
    )


    shapiro_result = stats.shapiro(
        haversine_distance_km
    )


shapiro_statistic = float(
    shapiro_result.statistic
)


shapiro_p_value = float(
    shapiro_result.pvalue
)


if captured_shapiro_warnings:

    shapiro_warning_text = (
        " | ".join(
            str(
                warning.message
            )
            for warning
            in captured_shapiro_warnings
        )
    )


else:

    shapiro_warning_text = (
        "No warning generated."
    )


# ============================================================
# 47. JARQUE-BERA NORMALITY TEST
# ============================================================

jarque_bera_result = stats.jarque_bera(
    haversine_distance_km
)


jarque_bera_statistic = float(
    jarque_bera_result.statistic
)


jarque_bera_p_value = float(
    jarque_bera_result.pvalue
)


# ============================================================
# 48. NORMALITY TEST DECISIONS
# ============================================================

if shapiro_p_value < ALPHA:

    shapiro_decision = (
        "Reject H0"
    )


else:

    shapiro_decision = (
        "Fail to reject H0"
    )


if jarque_bera_p_value < ALPHA:

    jarque_bera_decision = (
        "Reject H0"
    )


else:

    jarque_bera_decision = (
        "Fail to reject H0"
    )


# ============================================================
# 49. COMBINED NORMALITY INTERPRETATION
# ============================================================

if (
    shapiro_p_value < ALPHA
    and jarque_bera_p_value < ALPHA
):

    normality_interpretation = (
        "Both Shapiro-Wilk and Jarque-Bera reject the null "
        "hypothesis of normality. The Haversine distance "
        "distribution should not be considered normally "
        "distributed at the selected significance level."
    )


elif (
    shapiro_p_value >= ALPHA
    and jarque_bera_p_value >= ALPHA
):

    normality_interpretation = (
        "Neither Shapiro-Wilk nor Jarque-Bera rejects the "
        "null hypothesis of normality. The Haversine distance "
        "distribution is statistically compatible with a "
        "normal distribution at the selected significance level."
    )


else:

    normality_interpretation = (
        "Shapiro-Wilk and Jarque-Bera provide different "
        "normality decisions. The statistical tests should "
        "therefore be interpreted together with skewness, "
        "kurtosis, the histogram and the Q-Q plot."
    )


# ============================================================
# 50. NORMALITY SUMMARY TABLE
# ============================================================

normality_summary_table = pd.DataFrame({

    "TEST": [
        "Shapiro-Wilk",
        "Jarque-Bera"
    ],

    "STATISTIC": [
        shapiro_statistic,
        jarque_bera_statistic
    ],

    "P_VALUE": [
        shapiro_p_value,
        jarque_bera_p_value
    ],

    "ALPHA": [
        ALPHA,
        ALPHA
    ],

    "DECISION": [
        shapiro_decision,
        jarque_bera_decision
    ]
})


# ============================================================
# 51. CREATE HAVERSINE DISTANCE HISTOGRAM
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.hist(
    haversine_distance_km,
    bins=100
)


ax.axvline(
    distance_median,
    linestyle="--",
    label=(
        f"Median = "
        f"{distance_median:.2f} km"
    )
)


ax.axvline(
    distance_mean,
    linestyle=":",
    label=(
        f"Mean = "
        f"{distance_mean:.2f} km"
    )
)


ax.set_xlabel(
    "Haversine distance (km)"
)


ax.set_ylabel(
    "Number of observations"
)


ax.set_title(
    "Distribution of sender-receiver Haversine distance"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    DISTANCE_DISTRIBUTION_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 52. PREPARE Q-Q PLOT DATA
# ============================================================

qq_sample_size = min(
    QQ_SAMPLE_MAXIMUM,
    len(
        haversine_distance_km
    )
)


if len(
    haversine_distance_km
) > qq_sample_size:

    random_generator = (
        np.random.default_rng(
            RANDOM_STATE
        )
    )


    qq_indices = (
        random_generator.choice(
            len(
                haversine_distance_km
            ),
            size=qq_sample_size,
            replace=False
        )
    )


    qq_distance_data = (
        haversine_distance_km[
            qq_indices
        ]
    )


else:

    qq_distance_data = (
        haversine_distance_km
    )


# ============================================================
# 53. CREATE HAVERSINE DISTANCE Q-Q PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        8
    )
)


stats.probplot(
    qq_distance_data,
    dist="norm",
    plot=ax
)


ax.set_title(
    "Q-Q plot of Haversine distance"
)


ax.set_xlabel(
    "Theoretical normal quantiles"
)


ax.set_ylabel(
    "Observed distance quantiles"
)


fig.tight_layout()


fig.savefig(
    DISTANCE_QQ_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 54. CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 55. PREPARE HTML TABLES
# ============================================================

descriptive_statistics_html = (
    descriptive_statistics
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pearson_matrix_html = (
    pearson_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


spearman_matrix_html = (
    spearman_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


correlation_summary_html = (
    correlation_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "COEFFICIENT":
                lambda x:
                f"{x:.6f}"
        }
    )
)


if number_high_correlations > 0:

    high_correlations_html = (
        high_correlations
        .to_html(
            index=False,
            border=0,
            formatters={
                "CORRELATION":
                    lambda x:
                    f"{x:.6f}",

                "ABSOLUTE_CORRELATION":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


else:

    high_correlations_html = (
        "<p>No feature pairs reached the selected "
        "high-correlation threshold.</p>"
    )


distance_summary_html = (
    distance_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "VALUE":
                lambda x:
                f"{x:.6f}"
        }
    )
)


normality_summary_html = (
    normality_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "STATISTIC":
                lambda x:
                f"{x:.12g}",

            "P_VALUE":
                lambda x:
                f"{x:.12e}",

            "ALPHA":
                lambda x:
                f"{x:.2f}"
        }
    )
)


# ============================================================
# 56. CONVERT STATIC FIGURES TO BASE64
# ============================================================

pearson_correlation_base64 = (
    image_to_base64(
        PEARSON_CORRELATION_PATH
    )
)


spearman_correlation_base64 = (
    image_to_base64(
        SPEARMAN_CORRELATION_PATH
    )
)


distance_distribution_base64 = (
    image_to_base64(
        DISTANCE_DISTRIBUTION_PATH
    )
)


distance_qq_base64 = (
    image_to_base64(
        DISTANCE_QQ_PATH
    )
)


# ============================================================
# 57. MAP EXPORT STATUS
# ============================================================

if map_png_created:

    map_export_status = (
        "The interactive map and the static PNG map "
        "were created successfully."
    )


else:

    map_export_status = (
        "The interactive map was created successfully. "
        "Static PNG export was not available in the current "
        "environment."
    )


if direction_map_png_created:

    direction_map_export_status = (
        "The interactive directional map and the static PNG "
        "map were created successfully."
    )


else:

    direction_map_export_status = (
        "The interactive directional map was created "
        "successfully. Static PNG export was not available "
        "in the current environment."
    )


# ============================================================
# 58. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Continuous Geographic Features
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1300px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.map-container {{
    width: 100%;
    margin-top: 25px;
    margin-bottom: 45px;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Continuous Geographic Features
</h1>


<p>

Features analyzed:

</p>


<ul>

<li>{SEND_LAT}</li>
<li>{SEND_LONG}</li>
<li>{RECEIVE_LAT}</li>
<li>{RECEIVE_LONG}</li>

</ul>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete observations used in statistical analysis:</strong>
{analysis_observations}

<br>

<strong>Observations excluded because of missing or non-finite values:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
1. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     2. RANDOM 1% WORLD MAP
========================================================= -->


<h2>
2. Sender and receiver geographic locations
</h2>


<p>

A reproducible random sample representing
<strong>{MAP_SAMPLE_FRACTION * 100:.1f}%</strong>
of the total dataset is used only for this map.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Rows randomly selected</td>
<td>{map_sample_size}</td>
</tr>

<tr>
<td>Sender locations displayed</td>
<td>{sender_map_observations}</td>
</tr>

<tr>
<td>Receiver locations displayed</td>
<td>{receiver_map_observations}</td>
</tr>

<tr>
<td>Random state</td>
<td>{RANDOM_STATE}</td>
</tr>

</table>


<div class="note">

<strong style="color:#0000FF;">
Blue points
</strong>
represent sender registered locations.

<br><br>

<strong style="color:#FF0000;">
Red points
</strong>
represent receiver locations.

<br><br>

The sample is used only for visualization.

Statistical analyses use all complete observations.

</div>


<div class="map-container">

{world_map_html_fragment}

</div>


<p>

{map_export_status}

</p>


<!-- ========================================================
     3. DIRECTION MAP
========================================================= -->


<h2>
3. Sender-to-receiver geographic direction
</h2>


<p>

A smaller reproducible random sample representing
<strong>{DIRECTION_MAP_SAMPLE_FRACTION * 100:.1f}%</strong>
of the total dataset is used to display
individual transaction directions.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Rows initially sampled</td>
<td>{direction_map_sample_size}</td>
</tr>

<tr>
<td>Complete transactions displayed</td>
<td>{direction_map_observations}</td>
</tr>

<tr>
<td>Directional arrows displayed</td>
<td>{direction_arrow_count}</td>
</tr>

<tr>
<td>Random state</td>
<td>{DIRECTION_MAP_RANDOM_STATE}</td>
</tr>

</table>


<div class="note">

<strong style="color:#0000FF;">
Blue points
</strong>
represent sender registered locations.

<br><br>

<strong style="color:#FF0000;">
Red points
</strong>
represent receiver locations.

<br><br>

<strong style="color:#00A000;">
Green arrows
</strong>
start at the sender registered location and
point toward the receiver location for the
same transaction.

<br><br>

The arrows are used as an exploratory visual
representation of origin and destination.
They are not introduced as new modeling
features at this stage.

</div>


<div class="map-container">

{direction_map_html_fragment}

</div>


<p>

{direction_map_export_status}

</p>


<!-- ========================================================
     4. PEARSON CORRELATION
========================================================= -->


<h2>
4. Pearson correlation
</h2>


<p>

Pearson correlation measures the direction and
strength of linear association between the
geographic variables.

</p>


{pearson_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{pearson_correlation_base64}"
    alt="Pearson correlation matrix"
>

</div>


<!-- ========================================================
     5. SPEARMAN CORRELATION
========================================================= -->


<h2>
5. Spearman correlation
</h2>


<p>

Spearman correlation measures the direction and
strength of monotonic association using ranks.

Unlike Pearson correlation, the relationship
does not need to be strictly linear.

</p>


{spearman_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{spearman_correlation_base64}"
    alt="Spearman correlation matrix"
>

</div>


<!-- ========================================================
     6. CORRELATION AND REDUNDANCY
========================================================= -->


<h2>
6. Correlation and potential redundancy
</h2>


<h3>
Main sender-receiver relationships
</h3>


{correlation_summary_html}


<h3>
High-correlation pairs
</h3>


<p>

Pairs are flagged when the absolute correlation
is greater than or equal to:

<strong>{HIGH_CORRELATION_THRESHOLD:.2f}</strong>

</p>


{high_correlations_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

{redundancy_interpretation}

<br><br>

High correlation does not automatically mean
that a feature should be removed.

At this stage, correlation is used to identify
possible redundancy.

The decision to retain, transform, combine or
remove geographic variables should be made in
subsequent multicollinearity, PCA and modeling
analyses.

</div>


<!-- ========================================================
     7. HAVERSINE DISTANCE
========================================================= -->


<h2>
7. Haversine distance
</h2>


<p>

The Haversine formula combines latitude and
longitude to estimate the great-circle distance
between the sender registered location and the
receiver location.

A distance is calculated for every complete
geographic observation.

Distances are expressed in kilometers.

</p>


{distance_summary_html}


<div class="chart">

<img
    src="data:image/png;base64,{distance_distribution_base64}"
    alt="Haversine distance distribution"
>

</div>


<!-- ========================================================
     8. HAVERSINE DISTANCE NORMALITY
========================================================= -->


<h2>
8. Haversine distance normality
</h2>


<p>

The null hypothesis for both normality tests is
that the Haversine distance distribution is
compatible with a normal distribution.

</p>


{normality_summary_html}


<p class="result">

{normality_interpretation}

</p>


<table>

<tr>
<th>Distribution metric</th>
<th>Value</th>
</tr>

<tr>
<td>Skewness</td>
<td>{distance_skewness:.6f}</td>
</tr>

<tr>
<td>Excess kurtosis</td>
<td>{distance_kurtosis:.6f}</td>
</tr>

<tr>
<td>Q-Q plot observations</td>
<td>{qq_sample_size}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{distance_qq_base64}"
    alt="Haversine distance Q-Q plot"
>

</div>


<div class="note">

The Shapiro-Wilk test is calculated using the
complete Haversine distance array.

<br><br>

For very large samples, the Shapiro-Wilk test
can be extremely sensitive to small deviations
from normality and its p-value approximation
may be less accurate.

<br><br>

For this reason, normality should not be
evaluated only through the p-value.

Shapiro-Wilk, Jarque-Bera, skewness, kurtosis,
the histogram and the Q-Q plot should be
interpreted together.

<br><br>

Shapiro-Wilk message:

<br>

{shapiro_warning_text}

</div>


<!-- ========================================================
     9. SUMMARY
========================================================= -->


<h2>
9. Summary of results
</h2>


<p class="result">

Latitude Pearson correlation:
{latitude_pearson:.6f}
—
{latitude_pearson_strength},
{latitude_pearson_direction.lower()}.

</p>


<p class="result">

Latitude Spearman correlation:
{latitude_spearman:.6f}
—
{latitude_spearman_strength},
{latitude_spearman_direction.lower()}.

</p>


<p class="result">

Longitude Pearson correlation:
{longitude_pearson:.6f}
—
{longitude_pearson_strength},
{longitude_pearson_direction.lower()}.

</p>


<p class="result">

Longitude Spearman correlation:
{longitude_spearman:.6f}
—
{longitude_spearman_strength},
{longitude_spearman_direction.lower()}.

</p>


<p class="result">

Median sender-receiver Haversine distance:
{distance_median:.6f} km.

</p>


<p class="result">

Mean sender-receiver Haversine distance:
{distance_mean:.6f} km.

</p>


<p class="result">

Normality conclusion:

<br>

{normality_interpretation}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The geographic correlation analysis identifies
relationships and possible redundancy among the
original latitude and longitude features.

No geographic feature is removed or transformed
during this exploratory stage.

<br><br>

Potential redundancy identified here should be
revisited during multicollinearity analysis,
dimensionality reduction and GMM model
development.

<br><br>

The Haversine distance is included as an
exploratory derived measure because it summarizes
the physical separation between sender and
receiver.

<br><br>

The directional map provides an exploratory
visualization of the geographic movement from
sender location to receiver location.

No decision about introducing distance, bearing,
direction or other derived geographic features
into the final modeling dataset is made at this
stage.

</div>


</body>

</html>
"""


# ============================================================
# 59. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 60. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONTINUOUS GEOGRAPHIC - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


# ============================================================
# 61. DISPLAY FIRST MAP INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC LOCATION MAP"
)


print(
    "=" * 100
)


print(
    "\nRandom sample fraction:",
    f"{MAP_SAMPLE_FRACTION * 100:.1f}%"
)


print(
    "Rows randomly selected:",
    map_sample_size
)


print(
    "Sender locations displayed:",
    sender_map_observations
)


print(
    "Receiver locations displayed:",
    receiver_map_observations
)


print(
    "\nInteractive map:"
)


print(
    WORLD_MAP_HTML_PATH
)


# ============================================================
# 62. DISPLAY DIRECTION MAP INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SENDER-TO-RECEIVER DIRECTION MAP"
)


print(
    "=" * 100
)


print(
    "\nRandom sample fraction:",
    f"{DIRECTION_MAP_SAMPLE_FRACTION * 100:.1f}%"
)


print(
    "Rows initially sampled:",
    direction_map_sample_size
)


print(
    "Complete transactions displayed:",
    direction_map_observations
)


print(
    "Directional arrows displayed:",
    direction_arrow_count
)


print(
    "\nInteractive direction map:"
)


print(
    DIRECTION_MAP_HTML_PATH
)


# ============================================================
# 63. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS"
)


print(
    "=" * 100
)


display(
    descriptive_statistics
)


# ============================================================
# 64. DISPLAY PEARSON CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    pearson_matrix
)


# ============================================================
# 65. DISPLAY SPEARMAN CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SPEARMAN CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    spearman_matrix
)


# ============================================================
# 66. DISPLAY MAIN CORRELATIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MAIN SENDER-RECEIVER CORRELATIONS"
)


print(
    "=" * 100
)


display(
    correlation_summary_table
)


# ============================================================
# 67. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HIGH CORRELATIONS AND POTENTIAL REDUNDANCY"
)


print(
    "=" * 100
)


if number_high_correlations > 0:

    display(
        high_correlations
    )


else:

    print(
        "No feature pairs reached the selected threshold."
    )


print(
    "\nInterpretation:"
)


print(
    redundancy_interpretation
)


# ============================================================
# 68. DISPLAY HAVERSINE DISTANCE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HAVERSINE DISTANCE SUMMARY"
)


print(
    "=" * 100
)


display(
    distance_summary_table
)


# ============================================================
# 69. DISPLAY NORMALITY RESULTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HAVERSINE DISTANCE NORMALITY"
)


print(
    "=" * 100
)


display(
    normality_summary_table
)


print(
    "\nSkewness:",
    f"{distance_skewness:.6f}"
)


print(
    "Excess kurtosis:",
    f"{distance_kurtosis:.6f}"
)


print(
    "\nInterpretation:"
)


print(
    normality_interpretation
)


# ============================================================
# 70. DISPLAY STATIC MAP EXPORT STATUS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "STATIC MAP EXPORT STATUS"
)


print(
    "=" * 100
)


if map_png_created:

    print(
        "\nFirst map PNG:"
    )


    print(
        WORLD_MAP_PNG_PATH
    )


else:

    print(
        "\nFirst map PNG was not created."
    )


    if map_png_error:

        print(
            map_png_error
        )


if direction_map_png_created:

    print(
        "\nDirection map PNG:"
    )


    print(
        DIRECTION_MAP_PNG_PATH
    )


else:

    print(
        "\nDirection map PNG was not created."
    )


    if direction_map_png_error:

        print(
            direction_map_png_error
        )


# ============================================================
# 71. RELEASE MEMORY
# ============================================================

del dataset_geo
del complete_geo
del analysis_geo

del map_sample
del sender_map_sample
del receiver_map_sample

del direction_map_sample

del direction_send_lat
del direction_send_long
del direction_receive_lat
del direction_receive_long

del arrow_longitudes
del arrow_latitudes

del send_latitude
del send_longitude
del receive_latitude
del receive_longitude

del lat_1_rad
del lon_1_rad
del lat_2_rad
del lon_2_rad

del delta_latitude
del delta_longitude

del haversine_a
del central_angle
del haversine_distance_km

del qq_distance_data

gc.collect()


# ============================================================
# 72. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nInteractive geographic map:"
)


print(
    WORLD_MAP_HTML_PATH
)


print(
    "\nInteractive sender-to-receiver direction map:"
)


print(
    DIRECTION_MAP_HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    PEARSON_CORRELATION_PATH
)


print(
    SPEARMAN_CORRELATION_PATH
)


print(
    DISTANCE_DISTRIBUTION_PATH
)


print(
    DISTANCE_QQ_PATH
)


if map_png_created:

    print(
        WORLD_MAP_PNG_PATH
    )


if direction_map_png_created:

    print(
        DIRECTION_MAP_PNG_PATH
    )


CONTINUOUS GEOGRAPHIC - JOINT ANALYSIS

Total dataset observations: 1852394
Complete observations analyzed: 1852394
Excluded observations: 0

GEOGRAPHIC LOCATION MAP

Random sample fraction: 1.0%
Rows randomly selected: 18524
Sender locations displayed: 18524
Receiver locations displayed: 18524

Interactive map:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic/continuous_geographic_sender_receiver_map.html

SENDER-TO-RECEIVER DIRECTION MAP

Random sample fraction: 0.1%
Rows initially sampled: 1852
Complete transactions displayed: 1852
Directional arrows displayed: 1852

Interactive direction map:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic/continuous_geographic_sender_receiver_direction_map.html

DESCRIPTIVE STATISTICS


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max,variance,range
SEND_LAT_REGISTER,1852394.0,38.539310,5.071470,20.027100,26.472200,29.882601,31.770599,34.668900,39.354301,41.940399,44.447701,45.843300,48.478600,66.693298,25.719812,46.666199
SEND_LONG_REGISTER,1852394.0,-90.227837,13.747894,-165.672302,-123.061401,-119.082497,-111.098503,-96.797997,-87.476898,-80.157997,-74.978104,-73.536499,-70.345703,-67.950302,189.004608,97.722000
RECEIVE_LAT,1852394.0,38.538975,5.105604,19.027422,26.393135,29.753794,31.637751,34.740122,39.368900,41.956264,44.492343,46.002012,48.577546,67.510269,26.067190,48.482849
RECEIVE_LONG,1852394.0,-90.227943,13.759692,-166.671570,-123.557883,-119.309277,-111.244813,-96.899443,-87.440693,-80.245106,-74.937711,-73.365168,-70.397263,-66.950905,189.329132,99.720665



PEARSON CORRELATION MATRIX


,SEND_LAT_REGISTER,SEND_LONG_REGISTER,RECEIVE_LAT,RECEIVE_LONG
SEND_LAT_REGISTER,1.000000,-0.014744,0.993582,-0.014709
SEND_LONG_REGISTER,-0.014744,1.000000,-0.014585,0.999118
RECEIVE_LAT,0.993582,-0.014585,1.000000,-0.014554
RECEIVE_LONG,-0.014709,0.999118,-0.014554,1.000000



SPEARMAN CORRELATION MATRIX


...


Interpretation:
At least one geographic feature pair presents an absolute correlation equal to or greater than 0.90. This indicates potential redundancy among the geographic features. No feature is removed during this exploratory stage. These relationships should be considered later in multicollinearity analysis, dimensionality reduction and model construction.

HAVERSINE DISTANCE SUMMARY


,METRIC,VALUE
0,Minimum,0.022259
1,P1,11.126633
2,P5,24.756278
3,P10,34.987341
4,P25,55.320176
5,Median,78.216390
6,Mean,76.111831
7,P75,98.509760
8,P90,112.818065
9,P95,120.499805



HAVERSINE DISTANCE NORMALITY


,TEST,STATISTIC,P_VALUE,ALPHA,DECISION
0,Shapiro-Wilk,0.986496,4.912338e-101,0.05,Reject H0
1,Jarque-Bera,48187.347119,0.000000e+00,0.05,Reject H0



Skewness: -0.235677
Excess kurtosis: -0.634152

Interpretation:
Both Shapiro-Wilk and Jarque-Bera reject the null hypothesis of normality. The Haversine distance distribution should not be considered normally distributed at the selected significance level.

STATIC MAP EXPORT STATUS

First map PNG was not created.


Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



Direction map PNG was not created.


Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic

Main HTML report:
/projeto_t

## <span style="color:PURPLE"> CONTINUOS NUMERAL</span> ##

In [8]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "continuous_numeral"
)

AGE_FEATURE = (
    "SEND_AGE"
)

VALUE_FEATURE = (
    "TRANS_VALUE"
)

CONTINUOUS_NUMERAL_FEATURES = [
    AGE_FEATURE,
    VALUE_FEATURE
]

HIGH_CORRELATION_THRESHOLD = 0.90

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_continuous_numeral.html"
)


RELATIONSHIP_PATH = (
    RESULTS_DIRECTORY
    / "continuous_numeral_relationship.png"
)


RELATIONSHIP_LOG_PATH = (
    RESULTS_DIRECTORY
    / "continuous_numeral_relationship_log_value.png"
)


PEARSON_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_numeral_pearson_correlation.png"
)


SPEARMAN_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_numeral_spearman_correlation.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ONLY CONTINUOUS NUMERAL FEATURES
# ============================================================

dataset_continuous = pd.read_parquet(
    DATASET_PATH,
    columns=CONTINUOUS_NUMERAL_FEATURES
)


total_observations = int(
    len(
        dataset_continuous
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. KEEP COMPLETE FINITE OBSERVATIONS
#
# All calculations in this analysis use the same
# complete and finite observations.
#
# The original dataset is not modified.
# ============================================================

complete_continuous = (
    dataset_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_continuous = (
    complete_continuous.loc[
        finite_mask,
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_continuous
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 09. PREPARE NUMPY ARRAYS
# ============================================================

age_values = (
    analysis_continuous[
        AGE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


transaction_values = (
    analysis_continuous[
        VALUE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 10. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_statistics = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)


descriptive_statistics[
    "variance"
] = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .var()
)


descriptive_statistics[
    "range"
] = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .max()
    -
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .min()
)


descriptive_statistics[
    "iqr"
] = (
    descriptive_statistics[
        "75%"
    ]
    -
    descriptive_statistics[
        "25%"
    ]
)


# ============================================================
# 11. COVARIANCE MATRIX
#
# Covariance indicates whether the two variables
# tend to vary in the same or opposite direction.
#
# Its magnitude depends on the measurement units.
# ============================================================

covariance_matrix = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .cov()
)


age_value_covariance = float(
    covariance_matrix.loc[
        AGE_FEATURE,
        VALUE_FEATURE
    ]
)


if age_value_covariance > 0:

    covariance_direction = (
        "Positive"
    )


elif age_value_covariance < 0:

    covariance_direction = (
        "Negative"
    )


else:

    covariance_direction = (
        "Zero"
    )


# ============================================================
# 12. PEARSON CORRELATION MATRIX
#
# Pearson measures linear association.
# ============================================================

pearson_matrix = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .corr(
        method="pearson"
    )
)


pearson_correlation = float(
    pearson_matrix.loc[
        AGE_FEATURE,
        VALUE_FEATURE
    ]
)


# ============================================================
# 13. SPEARMAN CORRELATION MATRIX
#
# Spearman measures monotonic association using ranks.
# ============================================================

spearman_matrix = (
    analysis_continuous[
        CONTINUOUS_NUMERAL_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


spearman_correlation = float(
    spearman_matrix.loc[
        AGE_FEATURE,
        VALUE_FEATURE
    ]
)


# ============================================================
# 14. CORRELATION INTERPRETATION FUNCTION
# ============================================================

def interpret_correlation(
    value
):

    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        strength = (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        strength = (
            "Weak"
        )


    elif absolute_value < 0.50:

        strength = (
            "Moderate"
        )


    elif absolute_value < 0.70:

        strength = (
            "Strong"
        )


    else:

        strength = (
            "Very strong"
        )


    if value > 0:

        direction = (
            "Positive"
        )


    elif value < 0:

        direction = (
            "Negative"
        )


    else:

        direction = (
            "No directional association"
        )


    return (
        strength,
        direction
    )


# ============================================================
# 15. INTERPRET PEARSON AND SPEARMAN
# ============================================================

(
    pearson_strength,
    pearson_direction
) = interpret_correlation(
    pearson_correlation
)


(
    spearman_strength,
    spearman_direction
) = interpret_correlation(
    spearman_correlation
)


# ============================================================
# 16. PEARSON VS SPEARMAN COMPARISON
# ============================================================

correlation_difference = abs(
    pearson_correlation
    - spearman_correlation
)


absolute_correlation_difference = abs(
    abs(
        pearson_correlation
    )
    -
    abs(
        spearman_correlation
    )
)


if absolute_correlation_difference < 0.05:

    correlation_comparison_interpretation = (
        "Pearson and Spearman coefficients have very similar "
        "magnitudes. This suggests that the estimated linear "
        "and monotonic relationships are broadly consistent."
    )


elif absolute_correlation_difference < 0.15:

    correlation_comparison_interpretation = (
        "Pearson and Spearman coefficients show some difference "
        "in magnitude. The relationship may contain mild "
        "non-linearity, asymmetry or sensitivity to extreme values."
    )


else:

    correlation_comparison_interpretation = (
        "Pearson and Spearman coefficients differ substantially "
        "in magnitude. This may indicate non-linearity, asymmetry "
        "or an important influence of extreme observations."
    )


# ============================================================
# 17. CORRELATION SUMMARY TABLE
# ============================================================

correlation_summary_table = pd.DataFrame({

    "METHOD": [
        "Pearson",
        "Spearman"
    ],

    "RELATIONSHIP": [
        f"{AGE_FEATURE} × {VALUE_FEATURE}",
        f"{AGE_FEATURE} × {VALUE_FEATURE}"
    ],

    "COEFFICIENT": [
        pearson_correlation,
        spearman_correlation
    ],

    "DIRECTION": [
        pearson_direction,
        spearman_direction
    ],

    "STRENGTH": [
        pearson_strength,
        spearman_strength
    ]
})


# ============================================================
# 18. POTENTIAL REDUNDANCY
#
# A correlation equal to or greater than 0.90 in
# absolute value is flagged as potential redundancy.
#
# No feature is removed during this exploratory stage.
# ============================================================

pearson_redundancy = (
    abs(
        pearson_correlation
    )
    >= HIGH_CORRELATION_THRESHOLD
)


spearman_redundancy = (
    abs(
        spearman_correlation
    )
    >= HIGH_CORRELATION_THRESHOLD
)


potential_redundancy = (
    pearson_redundancy
    or spearman_redundancy
)


if potential_redundancy:

    redundancy_interpretation = (
        f"At least one correlation coefficient between "
        f"{AGE_FEATURE} and {VALUE_FEATURE} has an absolute "
        f"value equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"This pair is therefore flagged as potentially "
        f"redundant. No feature is removed at this stage. "
        f"The result should be revisited during multicollinearity, "
        f"dimensionality-reduction and modeling analyses."
    )


else:

    redundancy_interpretation = (
        f"Neither Pearson nor Spearman correlation between "
        f"{AGE_FEATURE} and {VALUE_FEATURE} reaches the "
        f"absolute threshold of "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"No strong evidence of redundancy is identified "
        f"between these two continuous numerical features."
    )


redundancy_table = pd.DataFrame({

    "RELATIONSHIP": [
        f"{AGE_FEATURE} × {VALUE_FEATURE}"
    ],

    "PEARSON": [
        pearson_correlation
    ],

    "SPEARMAN": [
        spearman_correlation
    ],

    "THRESHOLD": [
        HIGH_CORRELATION_THRESHOLD
    ],

    "POTENTIAL_REDUNDANCY": [
        (
            "Yes"
            if potential_redundancy
            else "No"
        )
    ]
})


# ============================================================
# 19. CREATE STANDARD HEXBIN RELATIONSHIP PLOT
#
# All complete finite observations are used.
#
# Hexagonal bins summarize local observation density,
# avoiding the overplotting produced by millions of
# individual scatter points.
#
# bins="log" applies logarithmic scaling only to the
# color representing observation density.
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        8
    )
)


hexbin = ax.hexbin(

    age_values,

    transaction_values,

    gridsize=120,

    mincnt=1,

    bins="log"
)


ax.set_xlabel(
    AGE_FEATURE
)


ax.set_ylabel(
    VALUE_FEATURE
)


ax.set_title(
    "Relationship between sender age and transaction value"
)


ax.grid(
    alpha=0.20
)


colorbar = fig.colorbar(
    hexbin,
    ax=ax
)


colorbar.set_label(
    "Log-scaled observation density"
)


fig.tight_layout()


fig.savefig(
    RELATIONSHIP_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 20. PREPARE POSITIVE TRANSACTION VALUES FOR LOG-Y PLOT
#
# A logarithmic y-axis requires strictly positive values.
#
# This filtering is used only for the second visualization.
# It does not modify the dataset or statistical analyses.
# ============================================================

positive_value_mask = (
    transaction_values
    > 0
)


age_values_log_plot = (
    age_values[
        positive_value_mask
    ]
)


transaction_values_log_plot = (
    transaction_values[
        positive_value_mask
    ]
)


positive_value_observations = int(
    len(
        transaction_values_log_plot
    )
)


non_positive_value_observations = (
    analysis_observations
    - positive_value_observations
)


# ============================================================
# 21. CREATE HEXBIN WITH LOGARITHMIC TRANS_VALUE AXIS
#
# All observations with TRANS_VALUE > 0 are used.
#
# TRANS_VALUE itself is not transformed in the dataset.
# Only the plot axis is logarithmic.
# ============================================================

if positive_value_observations > 0:

    fig, ax = plt.subplots(
        figsize=(
            11,
            8
        )
    )


    hexbin_log = ax.hexbin(

        age_values_log_plot,

        transaction_values_log_plot,

        gridsize=120,

        mincnt=1,

        bins="log",

        yscale="log"
    )


    ax.set_xlabel(
        AGE_FEATURE
    )


    ax.set_ylabel(
        f"{VALUE_FEATURE} - logarithmic scale"
    )


    ax.set_title(
        "Relationship between sender age and transaction value "
        "- logarithmic value axis"
    )


    ax.grid(
        alpha=0.20
    )


    colorbar = fig.colorbar(
        hexbin_log,
        ax=ax
    )


    colorbar.set_label(
        "Log-scaled observation density"
    )


    fig.tight_layout()


    fig.savefig(
        RELATIONSHIP_LOG_PATH,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


else:

    raise ValueError(
        "No positive transaction values are available "
        "for the logarithmic visualization."
    )


# ============================================================
# 22. FUNCTION TO CREATE CORRELATION MATRIX PLOT
# ============================================================

def create_correlation_plot(
    correlation_matrix,
    title,
    output_path
):

    values = (
        correlation_matrix
        .to_numpy(
            dtype="float64"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            7
        )
    )


    image = ax.imshow(
        values,
        vmin=-1,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                CONTINUOUS_NUMERAL_FEATURES
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                CONTINUOUS_NUMERAL_FEATURES
            )
        )
    )


    ax.set_xticklabels(
        CONTINUOUS_NUMERAL_FEATURES,
        rotation=35,
        ha="right"
    )


    ax.set_yticklabels(
        CONTINUOUS_NUMERAL_FEATURES
    )


    for row_index in range(
        len(
            CONTINUOUS_NUMERAL_FEATURES
        )
    ):

        for column_index in range(
            len(
                CONTINUOUS_NUMERAL_FEATURES
            )
        ):

            ax.text(
                column_index,
                row_index,
                (
                    f"{values[row_index, column_index]:.4f}"
                ),
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label="Correlation coefficient"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 23. CREATE PEARSON CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    pearson_matrix,
    "Pearson correlation matrix - continuous numerical features",
    PEARSON_CORRELATION_PATH
)


# ============================================================
# 24. CREATE SPEARMAN CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    spearman_matrix,
    "Spearman correlation matrix - continuous numerical features",
    SPEARMAN_CORRELATION_PATH
)


# ============================================================
# 25. CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 26. PREPARE HTML TABLES
# ============================================================

descriptive_statistics_html = (
    descriptive_statistics
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


covariance_matrix_html = (
    covariance_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pearson_matrix_html = (
    pearson_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


spearman_matrix_html = (
    spearman_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


correlation_summary_html = (
    correlation_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "COEFFICIENT":
                lambda x:
                f"{x:.6f}"
        }
    )
)


redundancy_html = (
    redundancy_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "PEARSON":
                lambda x:
                f"{x:.6f}",

            "SPEARMAN":
                lambda x:
                f"{x:.6f}",

            "THRESHOLD":
                lambda x:
                f"{x:.2f}"
        }
    )
)


# ============================================================
# 27. CONVERT IMAGES TO BASE64
# ============================================================

relationship_base64 = (
    image_to_base64(
        RELATIONSHIP_PATH
    )
)


relationship_log_base64 = (
    image_to_base64(
        RELATIONSHIP_LOG_PATH
    )
)


pearson_correlation_base64 = (
    image_to_base64(
        PEARSON_CORRELATION_PATH
    )
)


spearman_correlation_base64 = (
    image_to_base64(
        SPEARMAN_CORRELATION_PATH
    )
)


# ============================================================
# 28. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Continuous Numerical Features
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1300px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Continuous Numerical Features
</h1>


<p>

Features analyzed:

</p>


<ul>

<li>{AGE_FEATURE}</li>
<li>{VALUE_FEATURE}</li>

</ul>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Observations excluded because of missing or non-finite values:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
1. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     2. COVARIANCE
========================================================= -->


<h2>
2. Covariance
</h2>


<p>

Covariance evaluates whether the two variables
tend to vary in the same or opposite direction.

Its magnitude depends on the measurement units,
so covariance should not be interpreted as a
standardized measure of association strength.

</p>


{covariance_matrix_html}


<p class="result">

Covariance between {AGE_FEATURE} and {VALUE_FEATURE}:

{age_value_covariance:.6f}

—
{covariance_direction}.

</p>


<!-- ========================================================
     3. PEARSON CORRELATION
========================================================= -->


<h2>
3. Pearson correlation
</h2>


<p>

Pearson correlation measures the direction and
strength of linear association between
{AGE_FEATURE} and {VALUE_FEATURE}.

</p>


{pearson_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{pearson_correlation_base64}"
    alt="Pearson correlation matrix"
>

</div>


<p class="result">

Pearson correlation:

{pearson_correlation:.6f}

—
{pearson_strength},
{pearson_direction.lower()}.

</p>


<!-- ========================================================
     4. SPEARMAN CORRELATION
========================================================= -->


<h2>
4. Spearman correlation
</h2>


<p>

Spearman correlation measures the direction and
strength of monotonic association using ranks.

It does not require the relationship to be
strictly linear.

</p>


{spearman_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{spearman_correlation_base64}"
    alt="Spearman correlation matrix"
>

</div>


<p class="result">

Spearman correlation:

{spearman_correlation:.6f}

—
{spearman_strength},
{spearman_direction.lower()}.

</p>


<!-- ========================================================
     5. PEARSON AND SPEARMAN COMPARISON
========================================================= -->


<h2>
5. Pearson and Spearman comparison
</h2>


{correlation_summary_html}


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Absolute difference between coefficient magnitudes</td>
<td>{absolute_correlation_difference:.6f}</td>
</tr>

<tr>
<td>Signed coefficient difference</td>
<td>{correlation_difference:.6f}</td>
</tr>

</table>


<div class="note">

{correlation_comparison_interpretation}

</div>


<!-- ========================================================
     6. HEXBIN RELATIONSHIP
========================================================= -->


<h2>
6. Joint relationship between age and transaction value
</h2>


<p>

The hexagonal density plot uses all complete and
finite observations.

The horizontal axis represents {AGE_FEATURE}.

The vertical axis represents {VALUE_FEATURE}.

The color intensity represents the local
concentration of observations using a logarithmic
density scale.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{relationship_base64}"
    alt="Relationship between sender age and transaction value"
>

</div>


<!-- ========================================================
     7. LOGARITHMIC VALUE VISUALIZATION
========================================================= -->


<h2>
7. Joint relationship with logarithmic transaction-value axis
</h2>


<p>

This visualization uses a logarithmic vertical
axis for {VALUE_FEATURE}.

The logarithmic scale is applied only to the
visualization and does not modify the original
variable.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Observations available for statistical analysis</td>
<td>{analysis_observations}</td>
</tr>

<tr>
<td>Positive transaction values displayed</td>
<td>{positive_value_observations}</td>
</tr>

<tr>
<td>Non-positive values excluded only from this visualization</td>
<td>{non_positive_value_observations}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{relationship_log_base64}"
    alt="Relationship between sender age and transaction value with logarithmic value axis"
>

</div>


<!-- ========================================================
     8. POTENTIAL REDUNDANCY
========================================================= -->


<h2>
8. Potential redundancy
</h2>


<p>

A relationship is flagged as potentially redundant
when at least one absolute correlation coefficient
is greater than or equal to:

<strong>{HIGH_CORRELATION_THRESHOLD:.2f}</strong>

</p>


{redundancy_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

{redundancy_interpretation}

<br><br>

Correlation alone is not used to remove a feature
during this exploratory stage.

Any decision involving feature removal,
transformation or dimensionality reduction should
be made later together with multicollinearity,
PCA and modeling analyses.

</div>


<!-- ========================================================
     9. SUMMARY
========================================================= -->


<h2>
9. Summary of results
</h2>


<p class="result">

Covariance:

{age_value_covariance:.6f}
—
{covariance_direction.lower()}.

</p>


<p class="result">

Pearson correlation:

{pearson_correlation:.6f}
—
{pearson_strength},
{pearson_direction.lower()}.

</p>


<p class="result">

Spearman correlation:

{spearman_correlation:.6f}
—
{spearman_strength},
{spearman_direction.lower()}.

</p>


<p class="result">

Potential redundancy:

{
    "Yes"
    if potential_redundancy
    else "No"
}.

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The joint analysis evaluates whether sender age
and transaction value vary together and whether
their relationship is linear or monotonic.

<br><br>

Pearson and Spearman are interpreted jointly
because a difference between the two coefficients
may indicate non-linearity, asymmetry or
sensitivity to extreme transaction values.

<br><br>

The hexagonal density plots provide a direct
visual representation of the joint distribution
without randomly sampling the dataset.

<br><br>

No continuous numerical feature is removed or
transformed during this exploratory stage.

</div>


</body>

</html>
"""


# ============================================================
# 29. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 30. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONTINUOUS NUMERICAL - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 31. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS"
)


print(
    "=" * 100
)


display(
    descriptive_statistics
)


# ============================================================
# 32. DISPLAY COVARIANCE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "COVARIANCE MATRIX"
)


print(
    "=" * 100
)


display(
    covariance_matrix
)


print(
    "\nCovariance between "
    f"{AGE_FEATURE} and {VALUE_FEATURE}:"
)


print(
    f"{age_value_covariance:.6f}"
)


print(
    "Direction:",
    covariance_direction
)


# ============================================================
# 33. DISPLAY PEARSON CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    pearson_matrix
)


print(
    "\nPearson correlation:"
)


print(
    f"{pearson_correlation:.6f}"
)


print(
    "Strength:",
    pearson_strength
)


print(
    "Direction:",
    pearson_direction
)


# ============================================================
# 34. DISPLAY SPEARMAN CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SPEARMAN CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    spearman_matrix
)


print(
    "\nSpearman correlation:"
)


print(
    f"{spearman_correlation:.6f}"
)


print(
    "Strength:",
    spearman_strength
)


print(
    "Direction:",
    spearman_direction
)


# ============================================================
# 35. DISPLAY PEARSON VS SPEARMAN COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON VS SPEARMAN"
)


print(
    "=" * 100
)


display(
    correlation_summary_table
)


print(
    "\nAbsolute difference between coefficient magnitudes:"
)


print(
    f"{absolute_correlation_difference:.6f}"
)


print(
    "\nInterpretation:"
)


print(
    correlation_comparison_interpretation
)


# ============================================================
# 36. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "POTENTIAL REDUNDANCY"
)


print(
    "=" * 100
)


display(
    redundancy_table
)


print(
    "\nInterpretation:"
)


print(
    redundancy_interpretation
)


# ============================================================
# 37. DISPLAY LOG-PLOT INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "LOGARITHMIC TRANS_VALUE VISUALIZATION"
)


print(
    "=" * 100
)


print(
    "\nPositive transaction values displayed:",
    positive_value_observations
)


print(
    "Non-positive values excluded only from this visualization:",
    non_positive_value_observations
)


# ============================================================
# 38. RELEASE MEMORY
# ============================================================

del dataset_continuous
del complete_continuous
del analysis_continuous

del age_values
del transaction_values

del age_values_log_plot
del transaction_values_log_plot

gc.collect()


# ============================================================
# 39. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    RELATIONSHIP_PATH
)


print(
    RELATIONSHIP_LOG_PATH
)


print(
    PEARSON_CORRELATION_PATH
)


print(
    SPEARMAN_CORRELATION_PATH
)


CONTINUOUS NUMERICAL - JOINT ANALYSIS

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

DESCRIPTIVE STATISTICS


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max,variance,range,iqr
SEND_AGE,1852394.0,52.884144,17.402901,21.59,23.33,28.690001,32.369999,39.369999,50.759998,64.059998,77.459999,86.82,98.190002,101.839996,302.860962,80.249996,24.689999
TRANS_VALUE,1852394.0,70.063567,159.253975,1.00,1.26,2.440000,4.100000,9.640000,47.450000,83.100000,136.330000,195.34,537.900000,28948.900000,25361.828481,28947.900000,73.460000



COVARIANCE MATRIX


,SEND_AGE,TRANS_VALUE
SEND_AGE,302.860967,-29.544747
TRANS_VALUE,-29.544747,25361.828481



Covariance between SEND_AGE and TRANS_VALUE:
-29.544747
Direction: Negative

PEARSON CORRELATION MATRIX


...


Spearman correlation:
-0.024141
Strength: Very weak or negligible
Direction: Negative

PEARSON VS SPEARMAN


,METHOD,RELATIONSHIP,COEFFICIENT,DIRECTION,STRENGTH
0,Pearson,SEND_AGE × TRANS_VALUE,-0.010660,Negative,Very weak or negligible
1,Spearman,SEND_AGE × TRANS_VALUE,-0.024141,Negative,Very weak or negligible



Absolute difference between coefficient magnitudes:
0.013481

Interpretation:
Pearson and Spearman coefficients have very similar magnitudes. This suggests that the estimated linear and monotonic relationships are broadly consistent.

POTENTIAL REDUNDANCY


,RELATIONSHIP,PEARSON,SPEARMAN,THRESHOLD,POTENTIAL_REDUNDANCY
0,SEND_AGE × TRANS_VALUE,-0.01066,-0.024141,0.9,No



Interpretation:
Neither Pearson nor Spearman correlation between SEND_AGE and TRANS_VALUE reaches the absolute threshold of 0.90. No strong evidence of redundancy is identified between these two continuous numerical features.

LOGARITHMIC TRANS_VALUE VISUALIZATION

Positive transaction values displayed: 1852394
Non-positive values excluded only from this visualization: 0

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_numeral

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_numeral/analysis_continuous_numeral.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_numeral/continuous_numeral_relationship.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_gr

## <span style="color:PURPLE"> CYCLICAL ENDING USING SINE AND COSINE </span> ##

## <span style="color:PURPLE"> DISCRETE_NUMERAL </span> ##

## <span style="color:PURPLE"> FREQUENCY ENCODING WITH FALLBACK </span> ##

## <span style="color:PURPLE"> ONEHOT ENCODING WITH IGNORE </span> ##